<a href="https://colab.research.google.com/github/Kirip/Regime-Dependent-Beta-Optimization/blob/main/Beta_Calculations_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files

In [ ]:
print("📂 Upload: updated3_2_All_Equities_Cleaned.xlsx  and  MSCI_Data.csv")
uploaded = files.upload()

df_equity = pd.read_excel("updated3.2_All_Equities_Cleaned.xlsx", parse_dates=["Dates"])
df_market = pd.read_csv("MSCI_Data.csv", parse_dates=["Date"])

df_equity.set_index("Dates", inplace=True)
df_market.set_index("Date",  inplace=True)

df_merged = df_market.join(df_equity, how="inner")

print(f"✅ Merged shape: {df_merged.shape}")
print(f"   Date range : {df_merged.index[0].date()} → {df_merged.index[-1].date()}")
df_merged.head(3)

📂 Upload: updated3_2_All_Equities_Cleaned.xlsx  and  MSCI_Data.csv


Saving MSCI_Data.csv to MSCI_Data (1).csv
Saving updated3.2_All_Equities_Cleaned.xlsx to updated3.2_All_Equities_Cleaned (1).xlsx
✅ Merged shape: (252, 1159)
   Date range : 1999-01-29 → 2019-12-31


,MSCI USA,MSCI USA Value,MSCI USA Low Size,MSCI USA Low Volatility,MSCI USA High Dividend Yield,MSCI USA Quality,MSCI USA Momentum,KATE US Equity,KBH US Equity,KDP US Equity,...,CTSH UW Equity,CTVA US Equity,CTXS UQ Equity,CTXS UW Equity,CVC US Equity,CVG US Equity,CVH US Equity,CVS US Equity,CVX US Equity,CXO US Equity
1999-01-29,2673.05,3277.734545,2384.21,995.479542,997.649417,683.831934,512.942863,19.0938,14.0938,NaN,...,0.8099,NaN,18.0409,18.0409,33.0573,18.0000,4.3333,27.375,37.2500,NaN
1999-02-26,2598.02,3251.106568,2391.18,969.079546,970.419053,658.588413,500.178311,16.8438,11.2500,NaN,...,0.8958,NaN,15.3534,15.3534,31.8920,17.3125,4.7639,26.500,38.4375,NaN
1999-03-31,2705.67,3332.419537,2344.66,989.782505,967.408227,682.563547,528.966699,16.3125,11.2813,NaN,...,0.5781,NaN,15.1793,15.1793,36.3692,17.1250,3.3333,23.750,44.3750,NaN


In [36]:
targets = [
    'MSCI USA',
    'MSCI USA Value', 'MSCI USA Low Size', 'MSCI USA Low Volatility',
    'MSCI USA High Dividend Yield', 'MSCI USA Quality', 'MSCI USA Momentum'
]

# Every column that isn't a target benchmark
equities = [col for col in df_merged.columns if col not in targets]

# Percent change on price levels → monthly returns
returns = df_merged[targets + equities].pct_change().dropna(how="all")


/tmp/ipykernel_183/948867976.py:11: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = df_merged[targets + equities].pct_change().dropna(how="all")


In [ ]:
# ──────────────────────────────────────────────────────────────
# Beta Calculation — Rolling Cov / Rolling Var (60-month window)
# β(i, m) = Cov(r_i, r_m) / Var(r_m)
# Same formula as standard OLS beta under rolling estimation
# ──────────────────────────────────────────────────────────────

WINDOW = 60

# Pre-compute rolling variance for all 7 targets at once — efficient
rolling_var = returns[targets].rolling(window=WINDOW).var()

# Storage: one DataFrame per equity
all_equities_betas = {}

for equity in equities:

    # Combine equity + targets into one slice for clean rolling
    combined     = returns[[equity] + targets]
    equity_betas = pd.DataFrame(index=returns.index)

    for target in targets:

        # Rolling covariance between this equity and this target
        rolling_cov = combined[equity].rolling(window=WINDOW).cov(combined[target])

        # β = Cov(r_equity, r_target) / Var(r_target)
        equity_betas[f'Beta_{target}'] = rolling_cov / rolling_var[target]

    # Drop first 59 months — not enough data for a full window
    equity_betas.dropna(inplace=True)

    all_equities_betas[equity] = equity_betas

all_equities_betas[equities[0]].describe().round(3)

,Beta_MSCI USA,Beta_MSCI USA Value,Beta_MSCI USA Low Size,Beta_MSCI USA Low Volatility,Beta_MSCI USA High Dividend Yield,Beta_MSCI USA Quality,Beta_MSCI USA Momentum
count,192.000,192.000,192.000,192.000,192.000,192.000,192.000
mean,1.667,1.562,0.475,1.954,1.649,1.751,1.271
std,0.699,0.613,0.673,0.757,0.636,0.808,0.732
min,0.507,0.596,-0.750,0.698,0.710,0.415,0.086
25%,0.992,0.907,-0.061,1.163,0.962,1.067,0.779
50%,1.709,1.721,0.255,1.989,1.692,1.687,0.974
75%,2.243,2.123,1.098,2.545,2.164,2.364,1.862
max,3.067,2.860,1.896,3.626,3.198,3.327,2.920


In [ ]:
# ANS CODE
# Beta Calculation

# 1. Define our targets (Benchmark + Factors)
targets = [
    'MSCI USA',
    'MSCI USA Value', 'MSCI USA Low Size', 'MSCI USA Low Volatility',
    'MSCI USA High Dividend Yield', 'MSCI USA Quality', 'MSCI USA Momentum'
]

# Identify equity columns (everything in the dataframe that isn't a target or the risk-free rate)
# Make sure to adjust 'RF_Rate' if your column is named differently or already dropped
equities = [col for col in returns.columns if col not in targets]

# 2. Calculate the 60-month rolling variance for ALL targets at once
rolling_vars = returns[targets].rolling(window=60).var()

# 3. Create a dictionary to store the results
all_betas = {}

# 4. Loop through each equity to calculate its specific Betas
for equity in equities:
    # Create an empty DataFrame to store this equity's betas
    equity_betas = pd.DataFrame(index=returns.index)

    for target in targets:
        # Calculate rolling covariance between the equity and the target
        rolling_cov = returns[equity].rolling(window=60).cov(returns[target])

        # Calculate Beta: Rolling Covariance / Rolling Variance of the target
        equity_betas[f'Beta_{target}'] = rolling_cov / rolling_vars[target]

    # Drop the initial 59 months with NaN
    equity_betas.dropna(inplace=True)

    # Store the completed DataFrame in our dictionary
    all_betas[equity] = equity_betas

all_betas[equities[0]].describe().round(3)

,Beta_MSCI USA,Beta_MSCI USA Value,Beta_MSCI USA Low Size,Beta_MSCI USA Low Volatility,Beta_MSCI USA High Dividend Yield,Beta_MSCI USA Quality,Beta_MSCI USA Momentum
count,192.000,192.000,192.000,192.000,192.000,192.000,192.000
mean,1.667,1.562,0.475,1.954,1.649,1.751,1.271
std,0.699,0.613,0.673,0.757,0.636,0.808,0.732
min,0.507,0.596,-0.750,0.698,0.710,0.415,0.086
25%,0.992,0.907,-0.061,1.163,0.962,1.067,0.779
50%,1.709,1.721,0.255,1.989,1.692,1.687,0.974
75%,2.243,2.123,1.098,2.545,2.164,2.364,1.862
max,3.067,2.860,1.896,3.626,3.198,3.327,2.920


In [35]:
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import time

# ── 1. Load & convert PRICES → RETURNS ──────────────────────────────────
msci_raw = pd.read_csv('MSCI_Data (1).csv')
msci_raw['Date'] = pd.to_datetime(msci_raw['Date'])
msci_raw = msci_raw.set_index('Date').sort_index()
msci_ret = msci_raw.pct_change(fill_method=None).dropna(how='all')

eq_raw = pd.read_excel('updated3.2_All_Equities_Cleaned (1).xlsx')
eq_raw = eq_raw.set_index('Dates').sort_index()
eq_ret = eq_raw.pct_change(fill_method=None).dropna(how='all')

# ── 2. Align dates & drop sparse tickers ────────────────────────────────
common_dates = msci_ret.index.intersection(eq_ret.index)   # 251 months
msci_ret = msci_ret.loc[common_dates]
eq_ret   = eq_ret.loc[common_dates]

eq_clean = eq_ret.loc[:, eq_ret.isna().mean() <= 0.50]    # 1022 equities

targets  = ['MSCI USA','MSCI USA Value','MSCI USA Low Size',
            'MSCI USA Low Volatility','MSCI USA High Dividend Yield',
            'MSCI USA Quality','MSCI USA Momentum']
equities = eq_clean.columns.tolist()
returns  = pd.concat([msci_ret[targets], eq_clean], axis=1)

# ── 3. Vectorized Downside Beta ──────────────────────────────────────────
WINDOW, MIN_OBS = 60, 10

tgt_arr = returns[targets].values.astype(float)   # (T, 7)
eq_arr  = returns[equities].values.astype(float)  # (T, 1022)
T, n_targets, n_eq = len(returns), len(targets), len(equities)
n_windows = T - WINDOW + 1
window_dates = returns.index[WINDOW - 1:]

tgt_wins = sliding_window_view(tgt_arr, WINDOW, axis=0)   # (n_windows, 7, 60)
eq_wins  = sliding_window_view(eq_arr,  WINDOW, axis=0)   # (n_windows, 1022, 60)
beta_arr = np.full((n_windows, n_eq, n_targets), np.nan)

t0 = time.time()
for w in range(n_windows):
    for t_idx in range(n_targets):
        tgt_w     = tgt_wins[w, t_idx, :]
        down_mask = (tgt_w < 0) & ~np.isnan(tgt_w)
        if down_mask.sum() < MIN_OBS: continue

        tgt_c = tgt_w[down_mask]; tgt_c = tgt_c - tgt_c.mean()
        var_t = (tgt_c ** 2).mean()
        if var_t < 1e-10: continue

        eq_d  = eq_wins[w][:, down_mask]        # ← two-step index: (n_eq, n_down)
        valid = ~np.isnan(eq_d).any(axis=1)
        if valid.sum() == 0: continue

        eq_sub = eq_d[valid]
        eq_c   = eq_sub - eq_sub.mean(axis=1, keepdims=True)
        beta_arr[w, valid, t_idx] = (eq_c * tgt_c).mean(axis=1) / var_t



all_equities_betas = {
    eq: pd.DataFrame(beta_arr[:, i, :], index=window_dates, columns=targets)
    for i, eq in enumerate(equities)
}
all_equities_betas[equities[0]].describe().round(3)

,MSCI USA,MSCI USA Value,MSCI USA Low Size,MSCI USA Low Volatility,MSCI USA High Dividend Yield,MSCI USA Quality,MSCI USA Momentum
count,192.000,192.000,192.000,192.000,192.000,192.000,192.000
mean,1.364,1.149,0.153,1.172,1.177,1.556,1.034
std,0.991,0.942,2.031,1.003,1.077,1.249,1.216
min,-1.661,-0.734,-3.681,-2.252,-0.796,-0.942,-1.350
25%,0.777,0.928,-1.176,0.848,0.832,0.679,-0.018
50%,1.377,1.218,-0.710,1.381,1.113,1.231,1.198
75%,2.046,1.480,2.539,1.737,1.271,2.344,1.602
max,3.646,3.511,3.342,3.718,4.367,4.491,3.918


In [47]:
import pandas as pd
import numpy as np
import time

# ── Load & align (same pipeline as before) ─────────────────────
msci_raw = pd.read_csv('MSCI_Data.csv')
msci_raw['Date'] = pd.to_datetime(msci_raw['Date'])
msci_raw = msci_raw.set_index('Date').sort_index()
msci_ret = msci_raw.pct_change(fill_method=None).dropna(how='all')

eq_raw = pd.read_excel('updated3.2_All_Equities_Cleaned (1).xlsx')
eq_raw = eq_raw.set_index('Dates').sort_index()
eq_ret = eq_raw.pct_change(fill_method=None).dropna(how='all')

common_dates = msci_ret.index.intersection(eq_ret.index)
msci_ret = msci_ret.loc[common_dates]
eq_ret   = eq_ret.loc[common_dates]
eq_clean = eq_ret.loc[:, eq_ret.isna().mean() <= 0.50]

targets  = ['MSCI USA','MSCI USA Value','MSCI USA Low Size',
            'MSCI USA Low Volatility','MSCI USA High Dividend Yield',
            'MSCI USA Quality','MSCI USA Momentum']
equities = eq_clean.columns.tolist()
returns  = pd.concat([msci_ret[targets], eq_clean], axis=1)

# ── Frazzini-Pedersen (2014) Beta ──────────────────────────
# β_FP = ρ(r_i, r_t) × (σ_i / σ_t)
#
# ρ  → 60-month rolling correlation  (long window: stable co-movement)
# σ  → 12-month rolling volatility   (short window: reactive to regime)
#
# Decoupling correlation from vol is the key innovation vs OLS beta.
# OLS betas spike during vol regimes even when true co-movement is unchanged.
# FP avoids this by letting each component update at its own natural frequency.
# ρ × (σ_i / σ_t)

CORR_WINDOW = 60
VOL_WINDOW  = 12

tgt_arr = returns[targets].values.astype(float)
eq_arr  = returns[equities].values.astype(float)
T, n_targets, n_eq = len(returns), len(targets), len(equities)
n_windows    = T - CORR_WINDOW + 1
window_dates = returns.index[CORR_WINDOW - 1:]

beta_arr = np.full((n_windows, n_eq, n_targets), np.nan)

t0 = time.time()
for w in range(n_windows):
    corr_start = w;  corr_end = w + CORR_WINDOW
    vol_start  = corr_end - VOL_WINDOW;  vol_end = corr_end

    for t_idx in range(n_targets):
        # Correlation (60-month)
        tgt_corr = tgt_arr[corr_start:corr_end, t_idx]
        if np.isnan(tgt_corr).any(): continue

        eq_corr  = eq_arr[corr_start:corr_end, :]
        tgt_c    = tgt_corr - tgt_corr.mean()
        eq_c     = eq_corr  - eq_corr.mean(axis=0)
        tgt_std  = np.sqrt((tgt_c**2).mean())
        eq_std   = np.sqrt((eq_c**2).mean(axis=0))
        valid    = (eq_std > 1e-10) & ~np.isnan(eq_std)

        rho = np.full(n_eq, np.nan)
        rho[valid] = (tgt_c @ eq_c[:, valid]) / (CORR_WINDOW * tgt_std * eq_std[valid])

        # Volatility (12-month)
        tgt_vol_w = tgt_arr[vol_start:vol_end, t_idx]
        if np.isnan(tgt_vol_w).any(): continue
        sigma_t = tgt_vol_w.std(ddof=1)
        if sigma_t < 1e-10: continue
        sigma_i = np.nanstd(eq_arr[vol_start:vol_end, :], axis=0, ddof=1)

        # β_FP = ρ × (σ_i / σ_t)
        ok = valid & ~np.isnan(sigma_i) & ~np.isnan(rho)
        beta_arr[w, ok, t_idx] = rho[ok] * (sigma_i[ok] / sigma_t)


all_equities_betas_FP = {
    eq: pd.DataFrame(beta_arr[:, i, :], index=window_dates, columns=targets)
    for i, eq in enumerate(equities)
}
all_equities_betas_FP[equities[0]].describe().round(3)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


✅ 1.28s | 1,085,854 valid estimates


,MSCI USA,MSCI USA Value,MSCI USA Low Size,MSCI USA Low Volatility,MSCI USA High Dividend Yield,MSCI USA Quality,MSCI USA Momentum
count,192.000,192.000,192.000,192.000,192.000,192.000,192.000
mean,1.750,1.636,0.538,1.947,1.710,1.729,1.224
std,0.979,0.862,0.813,1.127,1.029,0.932,0.812
min,0.000,0.000,-0.833,0.000,0.000,0.000,0.000
25%,1.132,1.138,-0.076,1.120,1.064,1.111,0.581
50%,1.759,1.642,0.081,1.874,1.564,1.781,1.269
75%,2.404,2.247,1.166,2.793,2.470,2.390,1.910
max,4.726,3.610,3.005,4.554,4.930,3.986,2.791
